# Drowsiness Detection using MediaPipe Face Landmarks
This notebook detects driver drowsiness by analyzing eye blink patterns using MediaPipe face landmark detection.
- Detects faces in real-time video using MediaPipe
- Extracts 468 facial landmarks
- Calculates eye aspect ratio (Eye blink ratio)
- Classifies state: Active, Drowsy, or Sleeping

## Step 1: Import Libraries

In [8]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os
import time

## Step 2: Configure Dataset Path

In [9]:
# Define dataset base path
BASE_DATASET_PATH = '../dataset'  # Change this to your dataset directory

# Ensure dataset directory exists
if not os.path.exists(BASE_DATASET_PATH):
    os.makedirs(BASE_DATASET_PATH)
    print(f'Created dataset directory: {BASE_DATASET_PATH}')

print(f'Dataset path: {BASE_DATASET_PATH}')

Dataset path: ../dataset


## Step 3: Initialize MediaPipe Face Landmarker
- Load the Face Landmarker model from the local model folder
- Use MediaPipe Tasks API in VIDEO mode for real-time webcam input
- Detect facial landmarks and compute eye aspect ratio (EAR)

In [10]:
# Local imports so this cell can run independently
import os
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Configure local Face Landmarker model path
MODEL_PATH = '../model/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f'Face landmarker model not found: {MODEL_PATH}')

# Create MediaPipe Face Landmarker in VIDEO mode
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    )
face_landmarker = vision.FaceLandmarker.create_from_options(options)

# Initialize camera
cap = cv2.VideoCapture(0)
print('Face landmarker ready:', os.path.basename(MODEL_PATH))

Face landmarker ready: face_landmarker.task


## Step 4: Initialize State Variables
Track eye blink counts and current status:

In [11]:
# Status tracking variables
sleep = 0       # Counter for closed eyes frames
drowsy = 0      # Counter for partially closed eyes frames
active = 0      # Counter for open eyes frames

# Eye aspect ratio thresholds (tune if only one state appears)
EAR_THRESHOLD_OPEN = 0.30
EAR_THRESHOLD_DROWSY = 0.24

# BGR colors for each state shown on frame
STATUS_COLORS = {
    "SLEEPING !!!": (0, 0, 255),   # Red
    "DROWSY !": (0, 165, 255),      # Orange
    "ACTIVE :)": (0, 255, 0),       # Green
    "NO FACE": (160, 160, 160),     # Gray
}

status = "NO FACE"
color = STATUS_COLORS[status]

## Step 5: Define Helper Functions
### Function 1: Compute Euclidean Distance

In [12]:
def compute(ptA, ptB):
    """Calculate Euclidean distance between two points"""
    dist = np.linalg.norm(ptA - ptB)
    return dist

### Function 2: Calculate Eye Aspect Ratio (Blink Detection)
Determines if eye is open, partially closed, or closed.
- Returns: 0 (closed), 1 (drowsy), 2 (open)

In [13]:
def blinked(a, b, c, d, e, f, open_thr=EAR_THRESHOLD_OPEN, drowsy_thr=EAR_THRESHOLD_DROWSY):
    """
    Calculate eye aspect ratio from 6 landmark points and classify state.
    Returns (state, ratio) where state: 2=open, 1=drowsy, 0=closed.
    """
    up = compute(b, d) + compute(c, e)      # Sum of vertical distances
    down = compute(a, f)                    # Horizontal distance
    ratio = up / (2.0 * down)               # Eye aspect ratio
    if ratio > open_thr:
        return 2, ratio  # Eye fully open
    elif ratio > drowsy_thr:
        return 1, ratio  # Eye partially closed (drowsy)
    else:
        return 0, ratio  # Eye closed (sleeping)

## Step 6: Main Real-Time Detection Loop
Capture webcam frames, run MediaPipe Face Landmarker, and classify drowsiness from eye landmarks.

In [ ]:
# Main real-time loop using MediaPipe Face Landmarker Tasks API

# Eye indices for MediaPipe 468-landmark topology
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Default state when no face is detected
    status = "NO FACE"
    left_ear = right_ear = 0.0

    # Create MediaPipe image and run VIDEO-mode inference
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    timestamp_ms = int(frame_idx * (1000 / 30))
    result = face_landmarker.detect_for_video(mp_image, timestamp_ms)
    frame_idx += 1

    if result.face_landmarks:
        face_landmarks = result.face_landmarks[0]
        coords = np.array([[lm.x * w, lm.y * h] for lm in face_landmarks], dtype=np.float32)

        left = coords[LEFT_EYE]
        right = coords[RIGHT_EYE]

        left_state, left_ear = blinked(left[0], left[1], left[2], left[3], left[4], left[5])
        right_state, right_ear = blinked(right[0], right[1], right[2], right[3], right[4], right[5])

        if left_state == 0 or right_state == 0:
            sleep += 1
            drowsy = 0
            active = 0
            status = "SLEEPING !!!" if sleep > 6 else "DROWSY !"
        elif left_state == 1 or right_state == 1:
            sleep = 0
            active = 0
            drowsy += 1
            status = "DROWSY !" if drowsy > 6 else "ACTIVE :)"
        else:
            drowsy = 0
            sleep = 0
            active += 1
            status = "ACTIVE :)"

        # Draw eye points for visualization
        for p in left:
            cv2.circle(frame, (int(p[0]), int(p[1])), 1, (0, 255, 255), -1)
        for p in right:
            cv2.circle(frame, (int(p[0]), int(p[1])), 1, (0, 255, 255), -1)

    color = STATUS_COLORS.get(status, (255, 255, 255))
    cv2.putText(frame, status, (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)
    cv2.putText(frame, f"EYE L:{left_ear:.3f} R:{right_ear:.3f}", (50, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"eye open>{EAR_THRESHOLD_OPEN} drowsy>{EAR_THRESHOLD_DROWSY}", (50, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.imshow('Drowsiness Detection', frame)

    key = cv2.waitKey(1)
    if key == 27:
        break

cap.release()
face_landmarker.close()
cv2.destroyAllWindows()

KeyboardInterrupt: 

## How MediaPipe Face Landmarker Works

### Pipeline
1. Load `.task` model from the local `model` folder.
2. Run Face Landmarker in VIDEO mode for each frame.
3. Read eye landmark points from the 468-point mesh.
4. Compute Eye Aspect Ratio (EAR) and classify state.

### Eye Landmark Sets
- Left eye: `33, 160, 158, 133, 153, 144`
- Right eye: `362, 385, 387, 263, 373, 380`

### EAR Formula
```
EAR = (||p2 - p4|| + ||p3 - p5||) / (2 * ||p1 - p6||)
```

### State Thresholds (tunable)
- `EAR > 0.30`: Active
- `0.24 < EAR <= 0.30`: Drowsy
- `EAR <= 0.24`: Sleeping